<a href="https://colab.research.google.com/github/SHAIKSHOHEDKAMALUDDIN/Credit_Card_Fraud_Detection/blob/main/Credit_Card_Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install -q tensorflow>=2.15.0 scikit-learn>=1.3.0 pandas>=2.0.0 numpy>=1.24.0 matplotlib>=3.7.0 seaborn>=0.12.0 gradio>=4.20.0 plotly>=5.18.0 joblib>=1.3.0 imbalanced-learn>=0.11.0 kaggle>=1.6.0

In [ ]:
#!pip install -q gradio plotly kaggle imbalanced-learn

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json (from kaggle.com/settings -> Create New Token)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip -d .

Saving creditcard.csv to creditcard.csv
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:04<00:00, 16.7MB/s]

Archive:  creditcardfraud.zip
  inflating: ./creditcard.csv        


In [ ]:
%%writefile utils.py

"""
utils.py
--------
Shared helper functions used by train.py:
- evaluation metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
- confusion matrix plotting
- ROC curve plotting
- reconstruction error plotting (for the Autoencoder)
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def evaluate_predictions(y_true, y_pred, y_scores, model_name="Model"):
    """
    Computes and pretty-prints Accuracy, Precision, Recall, F1-score and
    ROC-AUC for a set of binary predictions, and returns them as a dict.

    y_true   : ground-truth labels (0 = normal, 1 = fraud)
    y_pred   : hard predictions (0/1) after thresholding
    y_scores : soft scores/probabilities (used for ROC-AUC)
    """
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_scores)
    except ValueError:
        auc = float("nan")

    print(f"\n================ {model_name} — Evaluation ================")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {auc:.4f}")
    print("\nDetailed classification report:")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Fraud"], zero_division=0))
    print("=============================================================")

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
    }


def plot_confusion_matrix(y_true, y_pred, model_name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"]
    )
    plt.title(f"Confusion Matrix — {model_name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{model_name.lower().replace(' ', '_')}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved confusion matrix -> {path}")


def plot_roc_curve(y_true, y_scores, model_name="Model"):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    auc = roc_auc_score(y_true, y_scores)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc:.4f})", linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {model_name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"roc_curve_{model_name.lower().replace(' ', '_')}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved ROC curve -> {path}")


def plot_reconstruction_error(errors, threshold, y_true=None):
    plt.figure(figsize=(7, 4))
    if y_true is not None:
        plt.hist(errors[y_true == 0], bins=50, alpha=0.6, label="Normal", color="steelblue", density=True)
        plt.hist(errors[y_true == 1], bins=50, alpha=0.6, label="Fraud", color="crimson", density=True)
    else:
        plt.hist(errors, bins=50, alpha=0.7, color="steelblue")
    plt.axvline(threshold, color="black", linestyle="--", label=f"Threshold = {threshold:.5f}")
    plt.xlabel("Reconstruction Error (MSE)")
    plt.ylabel("Density")
    plt.title("Autoencoder Reconstruction Error Distribution")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "autoencoder_reconstruction_error.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved reconstruction error plot -> {path}")


def plot_training_history(history, model_name="Model"):
    hist = history.history
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(hist.get("loss", []), label="train_loss")
    if "val_loss" in hist:
        axes[0].plot(hist["val_loss"], label="val_loss")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    metric_key = "accuracy" if "accuracy" in hist else None
    if metric_key:
        axes[1].plot(hist[metric_key], label="train_accuracy")
        if "val_accuracy" in hist:
            axes[1].plot(hist["val_accuracy"], label="val_accuracy")
        axes[1].set_title(f"{model_name} — Accuracy")
        axes[1].set_xlabel("Epoch")
        axes[1].legend()
    else:
        axes[1].axis("off")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"training_history_{model_name.lower().replace(' ', '_')}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved training history plot -> {path}")


def best_threshold_by_f1(y_true, y_scores):
    """
    Sweeps thresholds over y_scores and returns the threshold that maximizes
    F1-score, so reported metrics reflect the best operating point rather
    than an arbitrary 0.5 cut-off.
    """
    thresholds = np.unique(y_scores)
    if len(thresholds) > 200:
        thresholds = np.quantile(y_scores, np.linspace(0, 1, 200))

    best_t, best_f1 = 0.5, -1
    for t in thresholds:
        preds = (y_scores >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

Writing utils.py


In [ ]:
"""
====================================================================================
 AI Agent for Credit Card Fraud Detection and Financial Risk Analysis
====================================================================================
 Models   : Deep Neural Network (DNN, supervised classifier)
            Autoencoder (unsupervised anomaly detector)
 Dataset  : https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

 Run this AFTER utils.py is in the same folder/working directory in Colab,
 and AFTER creditcard.csv has been downloaded (see dataset step above).
 Then simply run:  !python train.py   (or paste this whole file into one cell)
====================================================================================
"""

import os
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from utils import (
    evaluate_predictions,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_reconstruction_error,
    plot_training_history,
    best_threshold_by_f1,
)

# ------------------------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------------------------
SEED = 42
CSV_CANDIDATES = ["creditcard.csv", "data/creditcard.csv", "/content/creditcard.csv"]
SAVED_MODELS_DIR = "saved_models"
TEST_SIZE = 0.2
VAL_SIZE = 0.1          # taken out of the training split
DNN_EPOCHS = 30
AE_EPOCHS = 40
BATCH_SIZE = 256

np.random.seed(SEED)
tf.random.set_seed(SEED)
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)


# ------------------------------------------------------------------------------------
# 1) LOAD DATA
# ------------------------------------------------------------------------------------
def load_dataset():
    for path in CSV_CANDIDATES:
        if os.path.exists(path):
            print(f"Loading dataset from: {path}")
            return pd.read_csv(path)
    raise FileNotFoundError(
        "Could not find creditcard.csv. Download it from Kaggle and place it in "
        "the working directory (or in a 'data/' subfolder)."
    )


# ------------------------------------------------------------------------------------
# 2) PREPROCESS
# ------------------------------------------------------------------------------------
def preprocess(df):
    df = df.copy()
    df.drop_duplicates(inplace=True)

    # Time and Amount are on very different scales from the PCA components (V1-V28),
    # so they get standardized. V1-V28 are already PCA outputs (roughly standardized).
    scaler = StandardScaler()
    df[["Time", "Amount"]] = scaler.fit_transform(df[["Time", "Amount"]])

    X = df.drop(columns=["Class"]).values.astype("float32")
    y = df["Class"].values.astype("int32")

    return X, y, scaler, list(df.drop(columns=["Class"]).columns)


# ------------------------------------------------------------------------------------
# 3) DNN CLASSIFIER
# ------------------------------------------------------------------------------------
def build_dnn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="DNN_Fraud_Classifier")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")],
    )
    return model


def train_dnn(X_train, y_train, X_val, y_val):
    print("\n>>> Training DNN classifier ...")
    model = build_dnn(X_train.shape[1])

    # Handle severe class imbalance (~0.17% fraud) with class weights.
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
    print(f"Class weights used to counter imbalance: {class_weight}")

    early_stop = callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=6, restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=DNN_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=2,
    )
    plot_training_history(history, model_name="DNN")
    return model


# ------------------------------------------------------------------------------------
# 4) AUTOENCODER (unsupervised anomaly detector)
# ------------------------------------------------------------------------------------
def build_autoencoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(20, activation="relu")(inputs)
    x = layers.Dense(14, activation="relu")(x)
    x = layers.Dense(7, activation="relu")(x)         # bottleneck
    x = layers.Dense(14, activation="relu")(x)
    x = layers.Dense(20, activation="relu")(x)
    outputs = layers.Dense(input_dim, activation="linear")(x)

    model = models.Model(inputs, outputs, name="Autoencoder_Anomaly_Detector")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return model


def train_autoencoder(X_train, y_train, X_val):
    print("\n>>> Training Autoencoder (on NORMAL transactions only) ...")
    # Key idea: the autoencoder only ever sees legitimate transactions during
    # training, so it learns to reconstruct "normal" patterns well. Fraudulent
    # transactions, being unusual, will have a much higher reconstruction error
    # -> that error becomes the anomaly score.
    X_train_normal = X_train[y_train == 0]

    model = build_autoencoder(X_train.shape[1])
    early_stop = callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True
    )

    history = model.fit(
        X_train_normal, X_train_normal,
        validation_data=(X_val, X_val),
        epochs=AE_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=2,
    )
    plot_training_history(history, model_name="Autoencoder")
    return model


def reconstruction_error(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=1)


# ------------------------------------------------------------------------------------
# 5) MAIN PIPELINE
# ------------------------------------------------------------------------------------
def main():
    df = load_dataset()
    print(f"Dataset shape: {df.shape}")
    print(f"Fraud cases: {df['Class'].sum()} / {len(df)} "
          f"({100 * df['Class'].mean():.4f}% of all transactions)")

    X, y, scaler, feature_names = preprocess(df)

    # Train / validation / test split (stratified, so fraud ratio is preserved everywhere)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=VAL_SIZE, stratify=y_train_full, random_state=SEED
    )
    print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

    results = {}

    # ---------------- DNN ----------------
    dnn = train_dnn(X_train, y_train, X_val, y_val)
    dnn_probs_test = dnn.predict(X_test, verbose=0).ravel()
    dnn_preds_test = (dnn_probs_test >= 0.5).astype(int)

    results["DNN"] = evaluate_predictions(y_test, dnn_preds_test, dnn_probs_test, model_name="DNN")
    plot_confusion_matrix(y_test, dnn_preds_test, model_name="DNN")
    plot_roc_curve(y_test, dnn_probs_test, model_name="DNN")

    # Also report the best achievable F1 by tuning the decision threshold,
    # since 0.5 is rarely optimal for such an imbalanced problem.
    best_t, best_f1 = best_threshold_by_f1(y_test, dnn_probs_test)
    print(f"[DNN] Best F1-optimized threshold = {best_t:.4f} -> F1 = {best_f1:.4f}")

    dnn.save(os.path.join(SAVED_MODELS_DIR, "dnn_model.keras"))

    # ---------------- Autoencoder ----------------
    autoencoder = train_autoencoder(X_train, y_train, X_val)

    val_errors = reconstruction_error(autoencoder, X_val)
    test_errors = reconstruction_error(autoencoder, X_test)

    # Threshold chosen to maximize F1 on the validation set (more robust than an
    # arbitrary percentile), then applied unchanged to the held-out test set.
    ae_threshold, _ = best_threshold_by_f1(y_val, val_errors)
    ae_preds_test = (test_errors >= ae_threshold).astype(int)

    results["Autoencoder"] = evaluate_predictions(
        y_test, ae_preds_test, test_errors, model_name="Autoencoder"
    )
    plot_confusion_matrix(y_test, ae_preds_test, model_name="Autoencoder")
    plot_roc_curve(y_test, test_errors, model_name="Autoencoder")
    plot_reconstruction_error(test_errors, ae_threshold, y_true=y_test)

    autoencoder.save(os.path.join(SAVED_MODELS_DIR, "autoencoder_model.keras"))

    # ---------------- Save scaler + metadata for the UI (app.py) ----------------
    joblib.dump(scaler, os.path.join(SAVED_MODELS_DIR, "scaler.pkl"))
    joblib.dump(
        {
            "feature_names": feature_names,
            "ae_threshold": float(ae_threshold),
            "dnn_threshold": float(best_t),
        },
        os.path.join(SAVED_MODELS_DIR, "metadata.pkl"),
    )

    # ---------------- Final summary ----------------
    print("\n\n=========================== FINAL SUMMARY ===========================")
    for name, m in results.items():
        print(f"{name:12s} | Acc: {m['accuracy']:.4f} | Prec: {m['precision']:.4f} | "
              f"Recall: {m['recall']:.4f} | F1: {m['f1']:.4f} | ROC-AUC: {m['roc_auc']:.4f}")
    print("=======================================================================")
    print(f"\nModels + scaler saved in '{SAVED_MODELS_DIR}/'. "
          f"Run 'python app.py' next to launch the interactive prediction UI.")


if __name__ == "__main__":
    main()

Loading dataset from: creditcard.csv
Dataset shape: (284807, 31)
Fraud cases: 492 / 284807 (0.1727% of all transactions)
Train: (204282, 30) | Val: (22698, 30) | Test: (56746, 30)

>>> Training DNN classifier ...
Class weights used to counter imbalance: {0: 0.500833570328819, 1: 300.41470588235296}
Epoch 1/30
798/798 - 13s - 17ms/step - accuracy: 0.9083 - auc: 0.9323 - loss: 0.3020 - precision: 0.0150 - recall: 0.8382 - val_accuracy: 0.9718 - val_auc: 0.9951 - val_loss: 0.1660 - val_precision: 0.0522 - val_recall: 0.9211
Epoch 2/30
798/798 - 3s - 3ms/step - accuracy: 0.9597 - auc: 0.9800 - loss: 0.1710 - precision: 0.0362 - recall: 0.9059 - val_accuracy: 0.9842 - val_auc: 0.9971 - val_loss: 0.0956 - val_precision: 0.0897 - val_recall: 0.9211
Epoch 3/30
798/798 - 3s - 4ms/step - accuracy: 0.9616 - auc: 0.9772 - loss: 0.1738 - precision: 0.0382 - recall: 0.9118 - val_accuracy: 0.9776 - val_auc: 0.9982 - val_loss: 0.1023 - val_precision: 0.0679 - val_recall: 0.9737
Epoch 4/30
798/798 - 2s

In [ ]:
"""
====================================================================================
 Interactive UI — AI Agent for Credit Card Fraud Detection
====================================================================================
 Run this AFTER train.py has finished (it needs the files saved in saved_models/).

 HOW TO RUN IN GOOGLE COLAB
     !python app.py
 Gradio will print a local URL and a public "*.gradio.live" URL — click either
 to open the app in your browser, right from Colab.

 WHAT THE APP DOES
   - Lets you upload a CSV of transactions (same columns as creditcard.csv, i.e.
     Time, V1..V28, Amount, and optionally Class) and get fraud predictions for
     every row from both models, combined into one "AI Agent" verdict.
   - Also has a Quick Demo tab: sample real rows from the dataset (fraud + normal)
     with one click, so you can see the models in action without uploading anything.
   - Shows Accuracy / Precision / Recall / F1 / ROC-AUC in a live report if the
     uploaded CSV includes the true 'Class' labels (great for demoing the project).
====================================================================================
"""

import os
import numpy as np
import pandas as pd
import joblib
import gradio as gr
import tensorflow as tf
import plotly.graph_objects as go

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)

SAVED_MODELS_DIR = "saved_models"
CSV_CANDIDATES = ["creditcard.csv", "data/creditcard.csv", "/content/creditcard.csv"]


# ------------------------------------------------------------------------------------
# LOAD ARTIFACTS
# ------------------------------------------------------------------------------------
def load_artifacts():
    dnn_path = os.path.join(SAVED_MODELS_DIR, "dnn_model.keras")
    ae_path = os.path.join(SAVED_MODELS_DIR, "autoencoder_model.keras")
    scaler_path = os.path.join(SAVED_MODELS_DIR, "scaler.pkl")
    meta_path = os.path.join(SAVED_MODELS_DIR, "metadata.pkl")

    for p in [dnn_path, ae_path, scaler_path, meta_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"Missing '{p}'. Run 'python train.py' first to train and save the models."
            )

    dnn = tf.keras.models.load_model(dnn_path)
    autoencoder = tf.keras.models.load_model(ae_path)
    scaler = joblib.load(scaler_path)
    meta = joblib.load(meta_path)
    return dnn, autoencoder, scaler, meta


DNN_MODEL, AE_MODEL, SCALER, META = load_artifacts()
FEATURE_NAMES = META["feature_names"]
AE_THRESHOLD = META["ae_threshold"]
DNN_THRESHOLD = META["dnn_threshold"]


def _load_sample_pool():
    for path in CSV_CANDIDATES:
        if os.path.exists(path):
            return pd.read_csv(path)
    return None


SAMPLE_POOL = _load_sample_pool()


# ------------------------------------------------------------------------------------
# CORE PREDICTION LOGIC
# ------------------------------------------------------------------------------------
def _prepare_features(df):
    df = df.copy()
    df = df[[c for c in FEATURE_NAMES if c in df.columns]] if set(FEATURE_NAMES).issubset(df.columns) \
        else df.reindex(columns=FEATURE_NAMES, fill_value=0)
    df[["Time", "Amount"]] = SCALER.transform(df[["Time", "Amount"]])
    return df.values.astype("float32")


def run_agent_on_dataframe(raw_df):
    """
    Runs both models over every row of raw_df (which must contain the original,
    un-scaled Time/V1..V28/Amount columns) and returns a results dataframe plus
    an aggregate metrics dict (only if a ground-truth 'Class' column is present).
    """
    has_labels = "Class" in raw_df.columns
    y_true = raw_df["Class"].values.astype(int) if has_labels else None

    X = _prepare_features(raw_df)

    dnn_probs = DNN_MODEL.predict(X, verbose=0).ravel()
    dnn_preds = (dnn_probs >= DNN_THRESHOLD).astype(int)

    X_recon = AE_MODEL.predict(X, verbose=0)
    ae_errors = np.mean(np.square(X - X_recon), axis=1)
    ae_preds = (ae_errors >= AE_THRESHOLD).astype(int)

    # AI Agent decision layer: flag as fraud if EITHER specialist model raises
    # an alarm (favors recall — catching fraud — which is the priority in this
    # domain, since missing fraud is costlier than a false alarm).
    agent_preds = ((dnn_preds == 1) | (ae_preds == 1)).astype(int)
    agent_confidence = np.maximum(dnn_probs, ae_errors / (ae_errors.max() + 1e-9))

    out = raw_df.copy().reset_index(drop=True)
    out["DNN_Fraud_Probability"] = np.round(dnn_probs, 5)
    out["DNN_Verdict"] = np.where(dnn_preds == 1, "FRAUD", "Normal")
    out["Autoencoder_Reconstruction_Error"] = np.round(ae_errors, 6)
    out["Autoencoder_Verdict"] = np.where(ae_preds == 1, "FRAUD", "Normal")
    out["AI_Agent_Final_Verdict"] = np.where(agent_preds == 1, "🚨 FRAUD", "✅ Normal")
    out["Agent_Confidence"] = np.round(agent_confidence, 4)

    metrics = None
    if has_labels:
        metrics = {
            "Accuracy": accuracy_score(y_true, agent_preds),
            "Precision": precision_score(y_true, agent_preds, zero_division=0),
            "Recall": recall_score(y_true, agent_preds, zero_division=0),
            "F1-Score": f1_score(y_true, agent_preds, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, dnn_probs) if len(np.unique(y_true)) > 1 else float("nan"),
        }
        cm = confusion_matrix(y_true, agent_preds)
    else:
        cm = None

    return out, metrics, cm


# ------------------------------------------------------------------------------------
# UI CALLBACKS
# ------------------------------------------------------------------------------------
def predict_from_csv(file):
    if file is None:
        return None, "Please upload a CSV file first.", None
    raw_df = pd.read_csv(file.name)

    missing = [c for c in FEATURE_NAMES if c not in raw_df.columns]
    if missing:
        return None, f"⚠️ Uploaded CSV is missing required columns: {missing}", None

    out, metrics, cm = run_agent_on_dataframe(raw_df)

    n_fraud = int((out["AI_Agent_Final_Verdict"] == "🚨 FRAUD").sum())
    summary = f"### ✅ Processed {len(out)} transactions — **{n_fraud} flagged as fraud**\n\n"

    fig = None
    if metrics is not None:
        summary += "#### 📊 Evaluation against ground-truth labels\n\n"
        summary += "| Metric | Score |\n|---|---|\n"
        for k, v in metrics.items():
            summary += f"| {k} | {v:.4f} |\n"
        fig = build_confusion_matrix_figure(cm)

    display_cols = [c for c in raw_df.columns if c not in FEATURE_NAMES or c == "Amount"] + [
        "DNN_Fraud_Probability", "DNN_Verdict",
        "Autoencoder_Reconstruction_Error", "Autoencoder_Verdict",
        "AI_Agent_Final_Verdict", "Agent_Confidence",
    ]
    display_cols = list(dict.fromkeys(display_cols))  # de-dupe, keep order

    return out[display_cols], summary, fig


def build_confusion_matrix_figure(cm):
    if cm is None:
        return None
    fig = go.Figure(data=go.Heatmap(
        z=cm, x=["Predicted: Normal", "Predicted: Fraud"], y=["Actual: Normal", "Actual: Fraud"],
        colorscale="Blues", showscale=False,
        text=cm, texttemplate="%{text}", textfont={"size": 18},
    ))
    fig.update_layout(title="Confusion Matrix (AI Agent Final Verdict)", height=380)
    return fig


def load_random_sample(kind):
    if SAMPLE_POOL is None:
        return None, "⚠️ creditcard.csv not found alongside app.py — can't pull a live sample. Upload a CSV instead."
    pool = SAMPLE_POOL[SAMPLE_POOL["Class"] == (1 if kind == "Fraud example" else 0)]
    row = pool.sample(1)
    out, metrics, _ = run_agent_on_dataframe(row)

    verdict = out["AI_Agent_Final_Verdict"].iloc[0]
    dnn_p = out["DNN_Fraud_Probability"].iloc[0]
    ae_e = out["Autoencoder_Reconstruction_Error"].iloc[0]
    amount = row["Amount"].iloc[0]
    actual = "Fraud" if row["Class"].iloc[0] == 1 else "Normal"

    msg = (
        f"### {verdict}\n\n"
        f"| Field | Value |\n|---|---|\n"
        f"| Actual label | **{actual}** |\n"
        f"| Transaction amount | ${amount:,.2f} |\n"
        f"| DNN fraud probability | {dnn_p:.4f} |\n"
        f"| Autoencoder reconstruction error | {ae_e:.6f} (threshold: {AE_THRESHOLD:.6f}) |\n"
    )
    return out, msg


# ------------------------------------------------------------------------------------
# BUILD THE UI
# ------------------------------------------------------------------------------------
CUSTOM_CSS = """
#title { text-align: center; }
.gradio-container { max-width: 1100px !important; margin: auto; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=CUSTOM_CSS, title="Fraud Detection AI Agent") as demo:
    gr.Markdown(
        "# 🛡️ AI Agent for Credit Card Fraud Detection\n"
        "### Deep Neural Network + Autoencoder working together as one fraud-detection agent",
        elem_id="title",
    )
    gr.Markdown(
        "Upload a batch of transactions, or try a live sample pulled straight from the dataset. "
        "Every prediction combines a **supervised DNN classifier** (learns fraud patterns directly) "
        "with an **unsupervised Autoencoder** (flags anything that looks statistically unusual), "
        "so a transaction is only cleared if *both* specialists agree it's normal."
    )

    with gr.Tab("📁 Batch CSV Prediction"):
        with gr.Row():
            file_input = gr.File(label="Upload transactions CSV (same columns as creditcard.csv)", file_types=[".csv"])
        run_btn = gr.Button("🔍 Run Fraud Detection", variant="primary")
        summary_md = gr.Markdown()
        cm_plot = gr.Plot(label="Confusion Matrix")
        results_table = gr.Dataframe(label="Per-transaction results", wrap=True)

        run_btn.click(fn=predict_from_csv, inputs=file_input, outputs=[results_table, summary_md, cm_plot])

    with gr.Tab("⚡ Quick Demo (live sample)"):
        gr.Markdown("No upload needed — pull a real transaction from the dataset and see the AI Agent verdict instantly.")
        with gr.Row():
            fraud_btn = gr.Button("🚨 Try a known FRAUD example")
            normal_btn = gr.Button("✅ Try a known NORMAL example")
        demo_msg = gr.Markdown()
        demo_table = gr.Dataframe(label="Details")

        fraud_btn.click(fn=lambda: load_random_sample("Fraud example"), outputs=[demo_table, demo_msg])
        normal_btn.click(fn=lambda: load_random_sample("Normal example"), outputs=[demo_table, demo_msg])

    gr.Markdown(
        "---\n"
        "*Models trained on the [Kaggle Credit Card Fraud dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). "
        "Run `train.py` before this app to (re)generate the models used here.*"
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=False)

/tmp/ipykernel_533/254611863.py:215: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=CUSTOM_CSS, title="Fraud Detection AI Agent") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3498504be5ba2fa6f1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
